# NCTBench-Pilot — Step 2: Text Extraction
EasyOCR for Bengali and image-based English PDFs.
PyMuPDF for text-native PDFs (none in this corpus).

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter
sys.path.insert(0, str(Path(r"E:\CSAA Project\pipeline")))
from config import MANIFEST_PATH, OCR_DIR, EXTRACTED_DIR

## Extraction Method by PDF

In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
pymupdf = sum(1 for e in manifest.values()
              if e.get("extraction_method") == "pymupdf")
easyocr = sum(1 for e in manifest.values()
              if e.get("extraction_method") == "easyocr")
print(f"PyMuPDF (text-native) : {pymupdf} PDFs")
print(f"EasyOCR (image-based) : {easyocr} PDFs")
print("\nNote: All 16 PDFs in this corpus are image-based.")
print("Both Bengali and English textbooks are scanned PDFs.")

## Run Extraction (skips already-done files)

In [ ]:
from step2_extract import run_extraction
run_extraction()

## Extraction Quality Analysis

In [ ]:
ocr_files = list(OCR_DIR.rglob("*.json"))
print(f"OCR result files: {len(ocr_files)}\n")
print(f"{'File':45s} {'Pages':6s} {'Non-empty':10s} {'Avg chars':10s}")
print("-" * 75)
for ocr_file in sorted(ocr_files):
    data = json.loads(ocr_file.read_text(encoding="utf-8"))
    pages = data.get("pages", [])
    non_empty = sum(
        1 for p in pages if len(p.get("text","").strip()) > 20
    )
    avg_chars = (
        sum(len(p.get("text","")) for p in pages)
        / max(len(pages), 1)
    )
    print(f"{ocr_file.name:45s} "
          f"{len(pages):6d} "
          f"{non_empty:10d} "
          f"{avg_chars:10.0f}")

## Sample Extracted Text

In [ ]:
bn_files = sorted(OCR_DIR.glob("class6_bn/*.json"))
en_files = sorted(OCR_DIR.glob("class6_en/*.json"))
if bn_files:
    data = json.loads(bn_files[0].read_text(encoding="utf-8"))
    pages = [p for p in data["pages"] if p.get("text","").strip()]
    if pages:
        print("=== BENGALI SAMPLE (Class 6) ===")
        print(pages[0]["text"][:400])
        print()
if en_files:
    data = json.loads(en_files[0].read_text(encoding="utf-8"))
    pages = [p for p in data["pages"] if p.get("text","").strip()]
    if pages:
        print("=== ENGLISH SAMPLE (Class 6) ===")
        print(pages[0]["text"][:400])